# Snap Distance Append
## Centroid-to-Network-Node Distance for all_access

**Tess Vu**

Appends two columns to `tess_all_access.csv`:

- `walk_snap_dist_m`, meters from each SAL centroid to its nearest walk-network node
- `drive_snap_dist_m`, same for the drive network

No 2SFCA or Dijkstra computation is re-run. The issue is loading the four `.graphml` files (~10–20 min total), the snap computation itself is instant.

Working CRS: EPSG:32735 (UTM 35S, meters).

- Input: `data/tess_all_access.csv`, `data/sal_w_ward_dedup/`, `data/networks/*.graphml`
- Output: `data/tess_all_access_w_snap.csv`

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())))

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
from src.config import load_config
from src.crs import CRS_UTM35S
from src.paths import NETWORKS, POP_PRED_FINAL, SAL_W_WARD_DEDUP, TESS_ALL_ACCESS, TESS_ALL_ACCESS_W_SNAP

SNAP_FLAG_M = load_config()["snap_flag_m"]

GRAPHML_PATHS = {
    "Gauteng_walk": NETWORKS / "network_gauteng_walk.graphml",
    "Gauteng_drive": NETWORKS / "network_gauteng_drive.graphml",
    "KwaZulu-Natal_walk": NETWORKS / "network_kwazulu_natal_walk.graphml",
    "KwaZulu-Natal_drive": NETWORKS / "network_kwazulu_natal_drive.graphml",
}
print(f"snap flag threshold: {SNAP_FLAG_M} m")

In [ ]:
all_access = pd.read_csv(TESS_ALL_ACCESS, low_memory=False)
all_access["EA_CODE"] = pd.to_numeric(all_access["EA_CODE"], errors="coerce").astype("Int64")

print(f"ALL_ACCESS LOADED: {len(all_access)} rows")
print(f"Columns with snap distance already: {[c for c in all_access.columns if 'snap' in c]}")

# Centroid computation matches the 2SFCA notebook: centroid after to_crs(32735).
sal_geo = gpd.read_file(SAL_W_WARD_DEDUP)
sal_geo["EA_CODE"] = pd.to_numeric(sal_geo["EA_CODE"], errors="coerce").astype("Int64")

sal_proj = sal_geo.to_crs(CRS_UTM35S)
sal_proj["centroid_x"] = sal_proj.geometry.centroid.x
sal_proj["centroid_y"] = sal_proj.geometry.centroid.y

print(f"SAL CENTROIDS COMPUTED: {len(sal_proj)} features  |  CRS: {sal_proj.crs}")

ALL_ACCESS INPUT: data/tess_all_access.csv
SAL SHAPEFILE: data/sal_w_ward_dedup/sal_w_ward_dedup.shp
PROJECTED CRS: EPSG:32735


In [ ]:
centroid_lookup = (
    sal_proj[["EA_CODE", "centroid_x", "centroid_y"]]
    .drop_duplicates(subset="EA_CODE")
    .set_index("EA_CODE")
)

# Helper columns for this notebook only; dropped before saving.
all_access = all_access.merge(centroid_lookup, on="EA_CODE", how="left")
print(f"CENTROID MERGE: {all_access['centroid_x'].notna().sum()}/{len(all_access)} SALs matched")

ALL_ACCESS LOADED: 38380 rows
Columns with snap distance already: []
SAL CENTROIDS COMPUTED: 38380 features
CRS: EPSG:32735


## Compute snap distances per province and mode

Each graph is loaded once, projected, and discarded. `ox.nearest_nodes()` uses a KDTree, no routing. Snap distance is straight-line Euclidean meters from the SAL centroid to its snapped node, in EPSG:32735.

In [ ]:
all_access["walk_snap_dist_m"] = np.nan
all_access["drive_snap_dist_m"] = np.nan

for graph_key, graphml_path in GRAPHML_PATHS.items():
    province, mode = graph_key.rsplit("_", 1)
    col_name = f"{mode}_snap_dist_m"

    print(f"LOADING GRAPH: {graph_key}")
    G_raw = ox.load_graphml(graphml_path)
    G = ox.project_graph(G_raw, to_crs=CRS_UTM35S)
    print(f"  Nodes: {G.number_of_nodes():,}  Edges: {G.number_of_edges():,}")

    node_coord_dict = {
        node_id: (data["x"], data["y"])
        for node_id, data in G.nodes(data=True)
    }

    province_mask = all_access["PR_NAME"] == province
    prov_df = all_access.loc[province_mask]
    print(f"  SALs in {province}: {len(prov_df)}")

    valid_mask = prov_df["centroid_x"].notna() & prov_df["centroid_y"].notna()
    prov_valid = prov_df.loc[valid_mask]

    snapped_node_ids = ox.nearest_nodes(
        G,
        prov_valid["centroid_x"].values,
        prov_valid["centroid_y"].values,
    )
    snapped_x = np.array([node_coord_dict[n][0] for n in snapped_node_ids])
    snapped_y = np.array([node_coord_dict[n][1] for n in snapped_node_ids])

    snap_dist = np.sqrt(
        (prov_valid["centroid_x"].values - snapped_x) ** 2 +
        (prov_valid["centroid_y"].values - snapped_y) ** 2
    )
    all_access.loc[prov_valid.index, col_name] = snap_dist

    print(f"  {col_name}: mean={snap_dist.mean():.0f}m, "
          f"median={np.median(snap_dist):.0f}m, "
          f"max={snap_dist.max():.0f}m, "
          f"flagged >{SNAP_FLAG_M}m: {(snap_dist > SNAP_FLAG_M).sum()}")

    del G, G_raw
    print()

print("SNAP DISTANCE COMPUTATION COMPLETE")

CENTROID MERGE: 38380/38380 SALs matched


In [ ]:
print("SNAP DISTANCE SUMMARY")
print()
for col in ["walk_snap_dist_m", "drive_snap_dist_m"]:
    mode = col.split("_")[0].capitalize()
    print(f"{mode} snap distances (meters):")
    for province in ["Gauteng", "KwaZulu-Natal"]:
        vals = all_access.loc[all_access["PR_NAME"] == province, col].dropna()
        print(f"  {province}:")
        print(f"    mean={vals.mean():.0f}, median={np.median(vals):.0f}, "
              f"p95={np.percentile(vals, 95):.0f}, max={vals.max():.0f}")
        for t in [100, 500, 1000]:
            print(f"    SALs with snap > {t}m: {(vals > t).sum()} ({(vals > t).mean() * 100:.1f}%)")
    print()

LOADING GRAPH: Gauteng_walk
  Nodes: 443,832  Edges: 1,219,312
  SALs in Gauteng: 20850
  walk_snap_dist_m: mean=83m, median=57m, max=25702m, flagged >500m: 232

LOADING GRAPH: Gauteng_drive
  Nodes: 279,004  Edges: 730,792
  SALs in Gauteng: 20850
  drive_snap_dist_m: mean=101m, median=67m, max=26962m, flagged >500m: 398

LOADING GRAPH: KwaZulu-Natal_walk
  Nodes: 541,204  Edges: 1,356,318
  SALs in KwaZulu-Natal: 17530
  walk_snap_dist_m: mean=226m, median=83m, max=9951m, flagged >500m: 2149

LOADING GRAPH: KwaZulu-Natal_drive
  Nodes: 311,426  Edges: 749,194
  SALs in KwaZulu-Natal: 17530
  drive_snap_dist_m: mean=302m, median=102m, max=8435m, flagged >500m: 2884

SNAP DISTANCE COMPUTATION COMPLETE


## DIAGNOSTIC SUMMARY

In [6]:
# Summarize snap distance distributions across provinces and modes.
print("SNAP DISTANCE SUMMARY")
print()

for col in ["walk_snap_dist_m", "drive_snap_dist_m"]:
    mode = col.split("_")[0].capitalize()
    print(f"{mode} snap distances (meters):")
    for province in ["Gauteng", "KwaZulu-Natal"]:
        mask = all_access["PR_NAME"] == province
        vals = all_access.loc[mask, col].dropna()
        print(f"  {province}:")
        print(f"    mean={vals.mean():.0f}, median={np.median(vals):.0f}, "
              f"p95={np.percentile(vals, 95):.0f}, max={vals.max():.0f}")
        print(f"    SALs with snap > 100m: {(vals > 100).sum()} "
              f"({(vals > 100).mean() * 100:.1f}%)")
        print(f"    SALs with snap > 500m: {(vals > 500).sum()} "
              f"({(vals > 500).mean() * 100:.1f}%)")
        print(f"    SALs with snap > 1000m: {(vals > 1000).sum()} "
              f"({(vals > 1000).mean() * 100:.1f}%)")
    print()

SNAP DISTANCE SUMMARY

Walk snap distances (meters):
  Gauteng:
    mean=83, median=57, p95=223, max=25702
    SALs with snap > 100m: 4467 (21.4%)
    SALs with snap > 500m: 232 (1.1%)
    SALs with snap > 1000m: 41 (0.2%)
  KwaZulu-Natal:
    mean=226, median=83, p95=985, max=9951
    SALs with snap > 100m: 7559 (43.1%)
    SALs with snap > 500m: 2149 (12.3%)
    SALs with snap > 1000m: 852 (4.9%)

Drive snap distances (meters):
  Gauteng:
    mean=101, median=67, p95=290, max=26962
    SALs with snap > 100m: 5949 (28.5%)
    SALs with snap > 500m: 398 (1.9%)
    SALs with snap > 1000m: 91 (0.4%)
  KwaZulu-Natal:
    mean=302, median=102, p95=1318, max=8435
    SALs with snap > 100m: 8929 (50.9%)
    SALs with snap > 500m: 2884 (16.5%)
    SALs with snap > 1000m: 1381 (7.9%)



In [ ]:
# High-snap cross-tab by settlement type.
pop_data = pd.read_csv(POP_PRED_FINAL)
pop_data["EA_CODE"] = pd.to_numeric(pop_data["EA_CODE"], errors="coerce").astype("Int64")
pop_subset = pop_data[["EA_CODE", "EA_GTYPE", "EA_TYPE"]].copy()

# Drop overlap cols first so the merge doesn't produce _x/_y suffixes.
overlap_cols = [c for c in ["EA_GTYPE", "EA_TYPE"] if c in all_access.columns]
diag = all_access.drop(columns=overlap_cols).merge(pop_subset, on="EA_CODE", how="left")

for col in ["walk_snap_dist_m", "drive_snap_dist_m"]:
    mode = col.split("_")[0]
    flag_col = f"{mode}_snap_flagged"
    diag[flag_col] = diag[col] > SNAP_FLAG_M

    print(f"{mode.upper()} SNAP FLAGS (>{SNAP_FLAG_M}m) BY SETTLEMENT TYPE")
    print(
        diag.groupby(["PR_NAME", "EA_GTYPE"])[flag_col]
        .agg(["sum", "count", "mean"])
        .rename(columns={"sum": "n_flagged", "count": "n_total", "mean": "pct_flagged"})
        .assign(pct_flagged=lambda x: (x["pct_flagged"] * 100).round(1))
        .to_string()
    )
    print()

WALK SNAP FLAGS (>500m) BY SETTLEMENT TYPE
                           n_flagged  n_total  pct_flagged
PR_NAME       EA_GTYPE                                    
Gauteng       Farms               91      321         28.3
              Traditional          4      256          1.6
              Urban              137    20273          0.7
KwaZulu-Natal Farms              454      758         59.9
              Traditional       1617     7935         20.4
              Urban               78     8837          0.9

DRIVE SNAP FLAGS (>500m) BY SETTLEMENT TYPE
                           n_flagged  n_total  pct_flagged
PR_NAME       EA_GTYPE                                    
Gauteng       Farms              119      321         37.1
              Traditional          4      256          1.6
              Urban              275    20273          1.4
KwaZulu-Natal Farms              571      758         75.3
              Traditional       2158     7935         27.2
              Urban        

In [ ]:
# Centroid helpers already live in sal_pharmacy_distances_k3.csv — drop before save.
all_access = all_access.drop(columns=["centroid_x", "centroid_y"], errors="ignore")

print("NEW COLUMNS ADDED: walk_snap_dist_m, drive_snap_dist_m")
print(f"walk_snap_dist_m non-null: {all_access['walk_snap_dist_m'].notna().sum()}")
print(f"drive_snap_dist_m non-null: {all_access['drive_snap_dist_m'].notna().sum()}")
print(f"Total rows: {len(all_access)}")

all_access.to_csv(TESS_ALL_ACCESS_W_SNAP, index=False)
print(f"SAVED: {TESS_ALL_ACCESS_W_SNAP}")

## NOTES

- Snap distance is the **Euclidean** distance from the SAL centroid to the nearest
  network node in the projected CRS (EPSG:32735, units = meters). It is not a
  routed or network distance.

- Walk and drive snap distances differ because the walk and drive graphs have
  different node sets (different streets qualify for each mode).

- A snap distance of zero means the centroid projected exactly onto a network
  node, which is rare. Small values (< 50m) are normal in dense urban areas.

- Large snap distances (> 500m) indicate either a genuine data gap in OSM for
  that area, or a large SAL whose geometric centroid falls far from any mapped
  road. These are the same SALs where the 2SFCA Ai scores and k-nearest
  distances carry the most measurement uncertainty.

- These columns do not replace the `walk_circuity_k1` and `drive_circuity_k1`
  columns in `sal_pharmacy_distances_k3.csv`. Circuity captures how tortuous
  the actual routed path is; snap distance captures how far the path even started
  from the true centroid position.